In [2]:
'''
task: classify syllogism validity with CGIF notation
models: gemma-2-2b-it
dataset: folio
evaluation: zero-shot + sef
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# benchmark experiment runtime
!pip install ipython-autotime
%load_ext autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.8 MB/s eta 0:00:00
time: 218 µs (started: 2026-04-23 07:41:00 +00:00)


In [4]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 84.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00
time: 26.4 s (started: 2026-04-23 07:41:00 +00:00)


In [5]:
import pandas as pd

folio_train_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_train_sef.csv")
folio_test_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_test_sef.csv")
folio_val_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/folio/data/folio_kr_gold_valid_sef.csv")
# merge splits into one dataframe
folio_df = pd.concat([folio_train_df, folio_test_df, folio_val_df], ignore_index=True, sort=False)
# check if merge worked
print("*** TRAIN SPLIT LEN:", len(folio_train_df))
print("*** TEST SPLIT LEN:", len(folio_test_df))
print("*** VALIDATION SPLIT LEN:", len(folio_val_df))
print("*** MERGED SPLIT LEN:", len(folio_df))

*** TRAIN SPLIT LEN: 800
*** TEST SPLIT LEN: 201
*** VALIDATION SPLIT LEN: 203
*** MERGED SPLIT LEN: 1204
time: 1.92 s (started: 2026-04-23 07:41:27 +00:00)


In [6]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, sef, ref_relation=None, source_knowledge=None):
  # define sef categories
  sef_disjunctive = r"""
  A disjunctive syllogism contains "∨" or "⊕". Here is an example:
  <PREMISES>[@every *x [(kid[(?x)]  young[(?x)])]
@every *x [(toddler[(?x)]  kid[(?x)])]
@every *x [(young[(?x)]  ~elderly[(?x)])]
@every *x [(pirate[(?x)]  seafarer[(?x)])]
~pirate[(nancy)]  young[(nancy)]
~toddler[(nancy)]  seafarer[(nancy)]]</PREMISES>
  <CONCLUSION>~[[(pirate[(nancy)]  toddler[(nancy)])]]</CONCLUSION>
  """

  sef_complex = r"""
  A complex syllogism has more than 2 premises. Here is an example:
  <PREMISES>[woncup[(aberdeen  year2013final)]
woncup[(rangers  year2014final)]
~[(aberdeen=rangers)]
@every *x @every *y @every *z @every *w [(~[(?x=y)]  woncup[(?x  z)]  woncup[(?y  w)]  ~[(?z=w)])]]</PREMISES>
  <CONCLUSION>[*x [(woncup[(aberdeen  x)])]]</CONCLUSION>
  """

  sef_categorical = r"""
  A categorical syllogism contains any word from the list ["all", "any", "some", "no", "few", "most", "none", "several"]. Here is an example:
  <PREMISES>[unincorporatedcommunity[(ordinary)]
locatedin[(ordinary  elliotcounty)]  on[(ordinary  kentuckyroute32)]
locatednorthwestof[(ordinary  sandyhook)]]</PREMISES>
  <CONCLUSION>[*x [(unincorporatedcommunity[(?x)]  locatedin[(?x  elliotcounty)])]]</CONCLUSION>
  """

  sef_hypothetical = r"""
  A hypothetical syllogism is not disjunctive, complex or categorical. Here is an example:
  <PREMISES>[@every *x [(homework[(?x)]  ~fun[(?x)])]
*x [(reading[(?x)]  homework[(?x)])]]</PREMISES>
  <CONCLUSION>[*x [(reading[(?x)]  fun[(?x)])]]</CONCLUSION>
  """

  sef_categories = dict()
  sef_categories["disjunctive"] = sef_disjunctive
  sef_categories["complex"] = sef_complex
  sef_categories["categorical"] = sef_categorical
  sef_categories["hypothetical"] = sef_hypothetical

  # define notation grammar
  grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline | keyword* quantifier* symbol* leftparen* (quantifier symbol)* proposition* rightparen* newline | keyword* (quantifier symbol)* leftparen* (quantifier symbol)* proposition rightparen* newline
    proposition: atomicproposition | complexproposition
    complexproposition: keyword* proposition keyword leftbracket* leftparen* (quantifier symbol)* proposition rightparen* rightbracket*
    atomicproposition: leftbracket* leftparen* term* leftbracket* leftparen* term* rightparen* rightbracket*
    !term: leftbracket* leftparen* quantifier* (LETTER+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠" | "?")* | (DIGIT+) (LETTER+|DIGIT+|"=" | "+" | "-" | "," | "≠" | "?")* rightparen* rightbracket*
    !leftbracket: "["
    !rightbracket: "]"
    !leftparen: "("
    !rightparen: ")"
    !keyword: "~" | "?"
    !quantifier: "*" | "@every *"
    symbol: LETTER
    newline: /\n/

    %import common.LETTER
    %import common.DIGIT
    %import common.INT -> NUMBER
    %import common.ESCAPED_STRING -> STRING
    %import common.WS
    %ignore WS
"""
  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in CGIF with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags.
  The CGIF BNF grammar to understand and reason in the language is given in the <GRAMMAR></GRAMMAR> tags.
  <GRAMMAR>{grammar}</GRAMMAR>
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  You are also given the category of the syllogism to help you understand it: {sef_categories[sef]}.
  Classify the conclusion as "True" if true, "False" if false or "Uncertain" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

time: 920 ms (started: 2026-04-23 07:41:28 +00:00)


In [7]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      if notation == "NL":
        conclusion = row["conclusion"]
        premises = row["premises"]
      else:
        conclusion = row["conclusion-" + notation]
        premises = row["premises-" + notation]
      sef_category = row["sef"]
      label = row["label"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, sef_category, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

time: 2.6 ms (started: 2026-04-23 07:41:29 +00:00)


In [8]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

time: 19.4 s (started: 2026-04-23 07:41:29 +00:00)


In [9]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

time: 4.21 s (started: 2026-04-23 07:41:49 +00:00)


In [11]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

time: 21.1 s (started: 2026-04-23 07:42:13 +00:00)


In [12]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(folio_df, model, tokenizer, mode='default', notation='CGIF')

Streaming output truncated to the last 5000 lines.
married[(katherinhafer  walterbrown)]]
*** Conclusion: 
 [graduatedwith[(walterbrown  bachelorsofart)]]
*** True Label: 
 True
*** Predicted Label: 
 True
*** Premises: 
 [@every *x [(convictedcriminal[(?x)]  innocent[(?x)]  ~trulyguilty[(?x)])]
@every *x [(convictedcriminal[(?x)]  ~commitcrime[(?x)]  innocent[(?x)])]
@every *x [(convictedcriminal[(?x)]  [(trulyguilty[(?x)]  foundguilty[(?x)])])]
@every *x [(convictedcriminal[(?x)]  foundguilty[(?x)]  sentencedtopunishment[(?x)])]
@every *x [(convictedcriminal[(?x)]  foundguilty[(?x)]  canargueagainst[(?x  punishment)])]
convictedcriminal[(garry)]  [(~[(foundguilty[(garry)]  sentencedtopunishment[(garry)])])]]
*** Conclusion: 
 [sentencedtopunishment[(garry)]]
*** True Label: 
 Uncertain
*** Predicted Label: 
 True
*** Premises: 
 [@every *x [(have[(?x  authorization  studyin  unitedstates)]  enrolledin[(?x  academicprogram)])]
@every *x [(enrolledin[(?x  academicprogram)]  ~work[(?x  

In [13]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.38870431893687707
***** PRECISION *****
0.38299880763116056
***** RECALL *****
0.3420558239398819
***** F1 *****
0.20806005148277443


,Accuracy,Precision,Recall,F1
0,0.388704,0.382999,0.342056,0.20806


time: 19.5 ms (started: 2026-04-23 08:21:33 +00:00)


In [14]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)

time: 17.4 ms (started: 2026-04-23 08:21:33 +00:00)
